# Preference VS Utility Benchmarks

This notebook includes the reproduction of table 3 and 4

In [1]:
import pandas as pd
import json
from scipy.stats import ttest_1samp
from statsmodels.stats.multicomp import pairwise_tukeyhsd

## Load Data

In [11]:
# Load data
rows = []
with open("../../data/beerAdvocate/all_results.jsonl", "r") as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)

metrics = ["rpp", "invrpp", "dcgrpp", "ap", "rbp", "rr", "ndcg", "rp"]

# Filter for preference rows only
# each row contains System A vs System B deltas for every query
df_pref = df[df['type'] == 'preference'].copy()

# Group by the pairs
pairs = df_pref.groupby(['runi', 'runj'])

results_table = []

for (run_i, run_j), group in pairs:
    pair_results = {"pair": f"{run_i}:{run_j}"}

    for m in metrics:
        # Distribution of differences (eg. SystemA_NDCG - SystemB_NDCG)
        scores = group[m].dropna()

        t_stat, p_val = ttest_1samp(scores, 0)
        
        pair_results[f"{m}_p"] = p_val
        pair_results[f"{m}_mean"] = scores.mean()
        
    results_table.append(pair_results)

df_stats = pd.DataFrame(results_table)

## t-test + Bonferroni

The table is based on pairwise differences.

In [17]:
df_hard = df_stats[abs(df_stats['ndcg_mean']) < 0.01]

alpha = 0.05
num_pairs = len(df_hard)
bonferroni_threshold = alpha / num_pairs # keep the overall error rate at 0.05 across all pairs
extreme_threshold = 1e-10

print(f"Percentage of significant differences (t-test, Bonferroni @ {bonferroni_threshold:.6e}):")
for m in metrics:
    sig_count = (df_hard[f"{m}_p"] < bonferroni_threshold).sum()
    percentage = (sig_count / num_pairs) * 100
    print(f"{m:15}: {percentage:.2f}%")

Percentage of significant differences (t-test, Bonferroni @ 2.000000e-03):
rpp            : 76.00%
invrpp         : 84.00%
dcgrpp         : 76.00%
ap             : 68.00%
rbp            : 60.00%
rr             : 56.00%
ndcg           : 84.00%
rp             : 76.00%


## Tukey’s HSD

In [7]:
# Retrieve the absolute metric scores
metric_rows = []
with open("../../data/beerAdvocate/all_results.jsonl", "r") as f:
    for line in f:
        data = json.loads(line)
        if data.get("type") == "metric":
            metric_rows.append(data)

df_absolute = pd.DataFrame(metric_rows)

print("=== Table 3: Tukey's HSD Sensitivity ===")
for m in ["ap", "ndcg", "rr", "rp"]:
    if m in df_absolute.columns:
        df_absolute[m] = pd.to_numeric(df_absolute[m], errors='coerce')
            
        res = pairwise_tukeyhsd(endog=df_absolute[m], groups=df_absolute['run'], alpha=0.05)
            
        sig_count = res.reject.sum()
        total_pairs = len(res.reject)
        percentage = (sig_count / total_pairs) * 100
        print(f"{m:10}: {percentage:.2f}% of pairs are significantly different")

# For RPP and variants (using the preference p-values in df_stats) use p < 0.05 without Bonferroni to match Tukey's sensitivity level
num_pairs = len(df_stats)
for m in ["rpp", "invrpp", "dcgrpp"]:
    sig_count = (df_stats[f"{m}_p"] < 0.05).sum()
    percentage = (sig_count / num_pairs) * 100
    print(f"{m:10}: {percentage:.2f}% of pairs are significantly different")

=== Table 3: Tukey's HSD Sensitivity ===
ap        : 93.33% of pairs are significantly different
ndcg      : 92.86% of pairs are significantly different
rr        : 90.48% of pairs are significantly different
rp        : 92.86% of pairs are significantly different
rpp       : 96.67% of pairs are significantly different
invrpp    : 99.05% of pairs are significantly different
dcgrpp    : 98.10% of pairs are significantly different
